In [6]:
%reload_ext autoreload
%autoreload 2

# Imports

In [7]:
from kret_notebook import *  # NOTE import first
from kret_lgbm._core.lgbm_nb_imports import *
from kret_lightning._core.lightning_nb_imports import *
from kret_matplotlib._core.mpl_nb_imports import *
from kret_np_pd._core.np_pd_nb_imports import *
from kret_optuna._core.optuna_nb_imports import *
from kret_polars._core.polars_nb_imports import *
from kret_rosetta._core.rosetta_nb_imports import *
from kret_sklearn._core.sklearn_nb_imports import *
from kret_torch_utils._core.torch_nb_imports import *
from kret_tqdm._core.tqdm_nb_imports import *
from kret_type_hints._core.types_nb_imports import *
from kret_utils._core.utils_nb_imports import *

# from kret_wandb._core.wandb_nb_imports import *  # NOTE this is slow to import

In [8]:
from nba_timeout_impact.load_data_utils import NBADataLoader
from nba_timeout_impact.clean_pipeline import NBAStatsV3CleanPipeline
from nba_timeout_impact.constants import NBAConstants
from nba_timeout_impact.nba_dataset import NBADataset

In [9]:
UKS_CONSTANTS.DATA_DIR

PosixPath('/Users/Akseldkw/coding/data_kretsinger')

In [10]:
NBA_DATA_DIR = UKS_CONSTANTS.DATA_DIR / "NBA"
os.makedirs(NBA_DATA_DIR, exist_ok=True)

# Load Data

In [11]:
raise ValueError("Skip Loading Data, Jump to Pipeline section")

ValueError: Skip Loading Data, Jump to Pipeline section

In [ ]:
# nba_statsv3 = NBADataLoader.load_seasons(data_type="nbastatsv3", playoffs=False, seasons=None)
nba_statsv3 = pd.read_parquet(NBA_DATA_DIR / "nba_statsv3_regular_season.parquet")
nba_statsv3["IsPlayoff"] = False

In [ ]:
# nba_statsv3_po = NBADataLoader.load_seasons(data_type="nbastatsv3", playoffs=True, seasons=None)
nba_statsv3_po = pd.read_parquet(NBA_DATA_DIR / "nba_statsv3_playoffs.parquet")
nba_statsv3_po["IsPlayoff"] = True

In [ ]:
nba_statsv3_all = pd.concat([nba_statsv3, nba_statsv3_po], ignore_index=True)

In [ ]:
nba_statsv3_all.to_parquet(NBA_DATA_DIR / "nba_statsv3.parquet")

In [ ]:
# early = NBADataLoader.load_game_dates_from_api(seasons=[1996, 1997, 1998, 1999])
# early_po = NBADataLoader.load_game_dates_from_api(seasons=[1996, 1997, 1998, 1999], playoffs=True)
# dates_nba = NBADataLoader.load_game_dates()
# dates_nba_po = NBADataLoader.load_game_dates(playoffs=True)

nba_game_dates = pd.read_parquet(NBA_DATA_DIR / "nba_game_dates.parquet")
dates_all = nba_game_dates

In [ ]:
game_date_map = dates_all.set_index("GAME_ID")["game_date"]
nba_statsv3["game_date"] = nba_statsv3["gameId"].map(game_date_map)

# Implementation

In [ ]:
# nba_statsv3.to_parquet(NBA_DATA_DIR / "nba_statsv3_regular_season.parquet")

In [ ]:
# nba_statsv3_po.to_parquet(NBA_DATA_DIR / "nba_statsv3_playoffs.parquet")

In [ ]:
# reload = pd.read_parquet(NBA_DATA_DIR / "nba_statsv3_regular_season.parquet")

# Pipeline

In [12]:
from nba_timeout_impact.clean_pipeline import SortByDate, CategoricalConversion

In [21]:
pipeline = NBAStatsV3CleanPipeline(NBA_DATA_DIR)
clean = pipeline.run()

In [22]:
UKS_NP_PD.dtt([clean], how="tail", show_dims=True, n=20)

,game_date_ffill,gameId,actionId,actionType,subType,scoreHome,scoreAway,pointsTotal,period,game_seconds_elapsed,seconds_remaining,seconds_elapsed,teamTricode,shotResult,isFieldGoal,location,description,shotValue,IsPlayoff,game_date,playerNameI,actionNumber,clock,teamId,personId,playerName,xLegacy,yLegacy,shotDistance,videoAvailable
,datetime64[s],int64,int64,category,category,int64,int64,int64,int64,float64,float64,float64,category,category,int64,category,str,float64,bool,datetime64[s],category,int64,str,category,int64,category,int64,int64,int64,int64
18049783,2025-06-22,42400407,523,Rebound,Unknown,101,87,188,4,2824.600,55.400,664.600,IND,NaN,0,v,Mathurin REBOUND (Off:7 Def:6),0.000,True,2025-06-22,B. Mathurin,734,PT00M55.40S,1610612754,1631097,Mathurin,0,0,0,1
18049784,2025-06-22,42400407,524,Turnover,Step Out of Bounds Turnover,101,87,188,4,2824.600,55.400,664.600,IND,NaN,0,v,Mathurin Step Out of Bounds Turnover (P3.T23),0.000,True,2025-06-22,B. Mathurin,735,PT00M55.40S,1610612754,1631097,Mathurin,0,0,0,1
18049785,2025-06-22,42400407,525,Missed Shot,Jump Shot,101,87,188,4,2846.500,33.500,686.500,OKC,Missed,1,h,MISS Dort 3PT Jump Shot,3.000,True,2025-06-22,L. Dort,736,PT00M33.50S,1610612760,1629652,Dort,230,26,0,1
18049786,2025-06-22,42400407,526,Rebound,Unknown,101,87,188,4,2847.600,32.400,687.600,NaN,NaN,0,h,THUNDER Rebound,0.000,True,2025-06-22,NaN,737,PT00M32.40S,0,1610612760,NaN,0,0,0,1
18049787,2025-06-22,42400407,527,Foul,Loose Ball,101,87,188,4,2847.600,32.400,687.600,IND,NaN,0,v,Toppin L.B.FOUL (P2.PN) (S.Wright),0.000,True,2025-06-22,O. Toppin,738,PT00M32.40S,1610612754,1630167,Toppin,0,0,0,1
18049788,2025-06-22,42400407,528,Free Throw,Free Throw 1 of 2,102,87,189,4,2847.600,32.400,687.600,OKC,NaN,0,h,Holmgren Free Throw 1 of 2 (17 PTS),0.000,True,2025-06-22,C. Holmgren,740,PT00M32.40S,1610612760,1631096,Holmgren,0,0,0,1
18049789,2025-06-22,42400407,529,Substitution,NaN,102,87,189,4,2847.600,32.400,687.600,OKC,NaN,0,h,SUB: Jones FOR Gilgeous-Alexander,0.000,True,2025-06-22,S. Gilgeous-Alexander,741,PT00M32.40S,1610612760,1628983,Gilgeous-Alexander,0,0,0,0
18049790,2025-06-22,42400407,530,Substitution,NaN,102,87,189,4,2847.600,32.400,687.600,OKC,NaN,0,h,SUB: Mitchell FOR Dort,0.000,True,2025-06-22,L. Dort,742,PT00M32.40S,1610612760,1629652,Dort,0,0,0,0
18049791,2025-06-22,42400407,531,Substitution,NaN,102,87,189,4,2847.600,32.400,687.600,OKC,NaN,0,h,SUB: K. Williams FOR Jal. Williams,0.000,True,2025-06-22,J. Williams,743,PT00M32.40S,1610612760,1631114,Williams,0,0,0,0


In [23]:
pipeline.save(clean, NBA_DATA_DIR / "nba_statsv3_clean.parquet")

Saved 18,049,803 rows x 30 cols -> /Users/Akseldkw/coding/data_kretsinger/NBA/nba_statsv3_clean.parquet


PosixPath('/Users/Akseldkw/coding/data_kretsinger/NBA/nba_statsv3_clean.parquet')

In [20]:
clean.scoreAway.isna().sum()

np.int64(0)

In [ ]:
UKS_NP_PD.count_unique(clean)

,game_date_ffill,gameId,actionId,actionType,subType,scoreHome,scoreAway,pointsTotal,period,teamTricode,shotResult,isFieldGoal,location,description,shotValue,IsPlayoff,seconds_remaining,seconds_elapsed,game_date,playerNameI,actionNumber,clock,teamId,personId,playerName,xLegacy,yLegacy,shotDistance,videoAvailable
,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64
0,5869,37514,745,14,185,173,173,342,8,36,2,2,2,3784108,3,2,1261,1801,5869,2597,998,1261,31,3638,1887,539,929,93,2


In [ ]:
type(pd.to_datetime(clean.game_date))

pandas.Series

# Sanbox

In [ ]:
stacked_raw = pd.read_parquet(NBA_DATA_DIR / "nba_statsv3.parquet")

In [ ]:
f_gameid = stacked_raw.gameId == 29600001

In [ ]:
scoreAwayFix = stacked_raw.groupby("gameId")["scoreAway"].cummax()

In [ ]:
UKS_NP_PD.dtt([stacked_raw, scoreAwayFix], filter=f_gameid, how="head")

actionNumber 
 clock 
 period 
 teamId 
 teamTricode 
 personId 
 playerName 
 playerNameI 
 xLegacy 
 yLegacy 
 shotDistance 
 shotResult 
 isFieldGoal 
 scoreHome 
 scoreAway 
 pointsTotal 
 location 
 description 
 actionType 
 subType 
 videoAvailable 
 actionId 
 gameId 
 shotValue 
 IsPlayoff 
 
 
 
 int64 
 str 
 int64 
 int64 
 str 
 int64 
 str 
 str 
 int64 
 int64 
 int64 
 str 
 int64 
 float64 
 float64 
 int64 
 str 
 str 
 str 
 str 
 int64 
 int64 
 int64 
 float64 
 bool 
 
 
 
 
 0 
 1 
 PT12M00.00S 
 1 
 0 
 NaN 
 0 
 NaN 
 NaN 
 0 
 0 
 0 
 NaN 
 0 
 0.000 
 0.000 
 0 
 NaN 
 Start of 1st Period (11:15 PM EST) 
 period 
 start 
 0 
 1 
 29600001 
 NaN 
 False 
 
 
 1 
 2 
 PT12M00.00S 
 1 
 1610612738 
 BOS 
 442 
 Ellison 
 P. Ellison 
 0 
 0 
 0 
 NaN 
 0 
 0.000 
 0.000 
 0 
 h 
 Jump Ball Ellison vs. Longley: Tip to Harper 
 Jump Ball 
 NaN 
 0 
 2 
 29600001 
 NaN 
 False 
 
 
 2 
 4 
 PT11M39.00S 
 1 
 1610612741 
 CHI 
 23 
 Rodman 
 D. Rodman 
 0 
 0 
 0 
 Made 
 1 
 0.000 
 2.000 
 2 
 v 
 Rodman  Layup (2 PTS) (Longley 1 AST) 
 Made Shot 
 Layup Shot 
 0 
 3 
 29600001 
 NaN 
 False 
 
 
 3 
 5 
 PT11M39.00S 
 1 
 1610612738 
 BOS 
 442 
 Ellison 
 P. Ellison 
 0 
 0 
 0 
 NaN 
 0 
 0.000 
 0.000 
 0 
 h 
 Ellison S.FOUL (P1.T1) 
 Foul 
 Shooting 
 0 
 4 
 29600001 
 NaN 
 False 
 
 
 4 
 6 
 PT11M39.00S 
 1 
 1610612741 
 CHI 
 23 
 Rodman 
 D. Rodman 
 0 
 0 
 0 
 NaN 
 0 
 0.000 
 3.000 
 3 
 v 
 Rodman Free Throw 1 of 1 (3 PTS) 
 Free Throw 
 Free Throw 1 of 1 
 0 
 5 
 29600001 
 NaN 
 False 
 
 
 
 
 
 
 scoreAway 
 
 
 
 float64 
 
 
 
 
 0 
 0.000 
 
 
 1 
 0.000 
 
 
 2 
 2.000 
 
 
 3 
 2.000 
 
 
 4 
 3.000